# Week 5 - Categories, tables and grouping

**Social and Cultural Analytics (7AAVBCS5)**

**Question: how do recorded hire durations differ between classic bikes and e-bikes?**

Use the lecture's skills to investigate this question. You can refer to the live coding for
patterns, but the columns, categories and comparisons here are different.

Work through sections 1 to 3 in order. Sections 4 and 5 are optional. Most written answers need
only one sentence, and the task says when a second is wanted.
You do not need to write functions, draw plots or join datasets.

Copy the download's data and weeks folders into your module folder, merging their contents.
In Colab, upload this notebook and bike_trips_16-31Jul2025_with_borough.csv, then run setup.

## Setup and data

Run the supplied cell below. It imports pandas and loads the bike-hire file as `trips`.
You do not need to write the loader.

London's Santander Cycles are shared bikes collected from and returned to docking stations.
This extract contains **443,539 recorded hires starting on 16–31 July 2025**. One record is one hire.
It does not cover a full year or identify why someone travelled.

| Column | Meaning |
|---|---|
| `bike_model` | `CLASSIC` for a classic bike; `PBSC_EBIKE` for an e-bike |
| `duration_min` | Recorded hire duration in minutes |
| `weekday` | Day on which the hire started |
| `end_station` | Name of the docking station where the hire ended |

The model, weekday and station columns load as text; duration loads as a number.
Some durations are zero or extremely long. Keep these records: investigating unusual values
is a normal part of EDA, and we will return to them in section 3.

In [ ]:
import subprocess
from pathlib import Path

import pandas as pd

DATA_REPO = "https://github.com/markjhill/7AAVBCS5-data.git"


def data_path(name: str) -> Path:
    """Return a usable path to a module data file."""
    here = Path.cwd()
    for candidate in (here, *here.parents):
        for local in (candidate / "data" / name, candidate / name, candidate / Path(name).name):
            if local.exists():
                return local

    downloaded = Path("7AAVBCS5-data")
    if not downloaded.is_dir():
        print("No local data folder found -- downloading the module data (once)...")
        subprocess.run(
            ["git", "clone", "--depth", "1", DATA_REPO, str(downloaded)],
            check=True,
        )
    return downloaded / name


# Supplied load for the bike-hire overview; reused in Part III.
trips = pd.read_csv(data_path("tfl-bike-hire/bike_trips_16-31Jul2025_with_borough.csv"))

In [ ]:
trips[["bike_model", "duration_min", "weekday", "end_station"]].head()

## 1. Make categories for the question

1. Identify the nominal variable: which bike model was hired?
2. Create an ordinal variable: how long was the recorded hire?
3. Check the difference between possible labels and labels present in a subset.

Bike model is nominal: neither model has a higher rank. Duration is numerical, but we can
group its values into ordered bands to compare the mix of shorter and longer hires.

**Task 1.1.** Convert bike_model to category and display its possible labels.

Use .astype("category") and .cat.categories, as in the lecture.

In [ ]:
# Your code here.

Use these duration bands. They are teaching choices, not fare bands or official definitions.

| Label | Recorded duration in minutes |
|---|---|
| Under 15 | Below 15 |
| 15 to under 30 | At least 15 and below 30 |
| 30 or more | At least 30 |

A recorded zero falls in Under 15; that does not establish that it was a valid completed journey.

**Task 1.2.** Create duration_band using these boundaries. Convert it to category and set the three labels in increasing duration order.

Adapt the lecture's rate-band pattern: create a column, assign labels with .loc conditions, convert to category, then use .cat.set_categories([...], ordered=True). Use & for two conditions that must both hold. Work one step at a time.

In [ ]:
# Your code here.

Now a different kind of check, this time on where hires begin rather than how long they last.

The supplied cell converts `start_borough` to a category, as you did for bike model, and then
keeps only the hires that ended at Putney Bridge Station. `trips` itself is unchanged.

In [ ]:
trips["start_borough"] = trips["start_borough"].astype("category")

putney = trips.loc[trips["end_station"] == "Putney Bridge Station, Fulham"].copy()
print("hires ending at Putney Bridge Station:", len(putney))

**Task 1.3.** For putney, print the number of possible start_borough labels and the number actually observed.

Use len(... .cat.categories) for the possible labels and .nunique() for the observed ones.

In [ ]:
# Your code here.

**Task 1.4.** Remove the unused start_borough labels from putney, then display the labels that remain.

Assign .cat.remove_unused_categories() back to the column in putney.

In [ ]:
# Your code here.

**Task 1.5.** Did removing the unused labels remove any hires? Say what changed and what did not.

*Write one sentence here.*

## 2. Compare the mix of hire durations

1. Count the models and duration bands.
2. Cross-tabulate model and duration band.
3. Compare percentages within models and within duration bands.
4. State what each denominator means.

Return to the full `trips` table for every calculation in this section.

**Task 2.1.** Count hires for each bike model, then express the counts as percentages rounded to one decimal place.

Use .value_counts(); normalize=True changes counts to proportions. Multiply by 100.

In [ ]:
# Your code here.

**Task 2.2.** Count hires in each duration band, preserving the order you set.

Use .value_counts(sort=False), as in the lecture's ordered-band counts. This keeps category order rather than sorting by frequency.

In [ ]:
# Your code here.

**Task 2.3.** Make model_duration_counts with bike_model as rows and duration_band as columns. Display the full table.

pd.crosstab(first_column, second_column) counts records for each combination; the first argument supplies rows.

In [ ]:
# Your code here.

**Task 2.4.** Make a second crosstab with a Total row and column. Check its grand total against len(trips).

Add margins=True, margins_name="Total" to the crosstab arguments.

In [ ]:
# Your code here.

**Task 2.5.** Calculate row percentages as within_model. What mix of duration bands appears within each bike model? Display one decimal place.

normalize="index" divides each count by its row total. Each model's row should sum to 100%.

In [ ]:
# Your code here.

**Task 2.6.** Calculate column percentages as within_band. Of the hires in each duration band, what percentage used each model?

Change the normalisation argument to "columns". Each duration-band column should sum to 100%.

In [ ]:
# Your code here.

**Task 2.7.** Answer all three parts, using the tables you just made.

(a) Among e-bike hires, what percentage lasted 30 minutes or more?

(b) Of all hires lasting 30 minutes or more, what percentage used an e-bike?

(c) From Task 2.1, e-bikes are 17.7% of all hires. That is the share you would expect in every
duration band if model and duration were unrelated. Compare each column percentage in
`within_band` with 17.7%. In which band are e-bikes a smaller share than that, and in which are
they a larger share?

*Write your answers here. One or two sentences is enough.*

## 3. Compare numerical summaries

Do the models differ when we use the actual recorded minutes rather than bands?

1. Grouping variable: `bike_model`. Which models are we comparing?
2. Numerical variable: `duration_min`. What measurement are we summarising?
3. Calculation: count, sum, mean or median. Which question does each answer?
4. Interpretation: one summary per model. What does it allow us to say?

This section does not tell you which method to use. The lecture slide "Useful operations on a
GroupBy object" lists your options; choose from it and say why.

Start with all recorded durations. A published analysis would normally set aside the hires that
cannot be journeys, and state the rule it used for doing so. Task 3.6 works out what that rule
would change, which is how you would decide on one.

**Task 3.1.** Create model_groups by grouping trips on bike_model. Count the hires in each group with .size().

Use the same two-line pattern as the lecture, with a different grouping column.

In [ ]:
# Your code here.

**Task 3.2.** Which model has the longer typical hire? Pick one calculation from the GroupBy table, apply it to duration_min, and use a comment to say why you picked it.

There is more than one defensible answer. The point is to choose and justify.

In [ ]:
# Your code here.

**Task 3.3.** Now create model_summary with count, sum, mean and median of duration_min together. Display one decimal place.

Use .agg([...]). count counts non-missing durations; sum gives total recorded minutes; mean and median describe minutes per hire. A larger total can reflect more hires, not longer individual hires.

In [ ]:
# Your code here.

**Task 3.4.** Which model has the higher mean duration, and which has the higher median? Include the values in one sentence.

*Write one sentence here.*

**Task 3.5.** The mean and the median disagree. Choose an operation from the GroupBy table that shows you the shape of each model's durations, apply it, and use a comment to say what you can now see.

Look for an operation that reports more than one number per group.

In [ ]:
# Your code here.

**Task 3.6.** Count the two groups of unusual durations the lecture warned about, zero minutes and over 24 hours. Give the total for each group, and then the count for each bike model.

A condition on its own gives True and False values, and .sum() counts the True ones. For the split by model, filter with .loc first and then group.

In [ ]:
# Your code here.

**Task 3.7.** Now keep only the hires that are in neither group, and calculate the same summary as Task 3.3 on what is left.

Leave the original `trips` table as it is. This is a check on how much the answer depends on those records, not a decision to delete them.

It is easier to say which hires to keep than which to remove. Use & for two conditions that must both hold, as you did in Task 1.2.

In [ ]:
# Your code here.

**Task 3.8.** Compare the summary in Task 3.7 with the one in Task 3.3.

Which of the two averages would you report, and what rule would you have to state alongside it? Refer to the numbers.

*Write two or three sentences here.*

**Task 3.9.** E-bike hires have the higher median duration. E-bikes also have motor assistance, so
in principle they cover a given distance faster than a classic bike would.

Does that make the claim "e-bikes are used to travel further" more convincing or less
convincing? Explain your reasoning, and say what your answer assumes.

*Write two or three sentences here.*

**Task 3.10.** Name the one extra measurement that would let you separate how far a hire went from how long it took. What could you then say that you cannot say now?

*Write one or two sentences here.*

## 4. Optional: extend the model comparison

The core worksheet is finished. Try these only if you have time. Use the same methods;
no additional data or joins are needed.

**Task 4.1.** Group by both bike_model and weekday, then calculate median duration_min. Are the model differences identical on every day?

Give groupby a list of two column names, then select the numerical column and use .median(). The period contains different numbers of each weekday; these are medians per hire, not daily hire totals.

In [ ]:
# Your code here.

Your duration bands in Task 1.2 used boundaries we chose. The lecture also showed
`pd.qcut()`, which picks boundaries so that each band holds roughly the same number of records.
The two approaches answer the same question in different ways.

**Task 4.2.** Use pd.qcut() to put duration_min into three equal-sized bands. Print the boundaries it chose, then compare how many hires fall in each band with your fixed bands from Task 1.2.

Use pd.qcut(column, q=3, labels=[...], retbins=True), which returns the bands and the boundaries. Print the boundaries, then compare the band sizes using .value_counts(sort=False) on each.

In [ ]:
# Your code here.

**Task 4.3.** The two sets of bands put very different numbers of hires in the lowest band. Which set is correct, and what does your answer tell you about reporting band boundaries?

*Write one or two sentences here.*

## 5. Work with your own data

Optional, and separate from everything above. If you have a dataset in mind for your project,
use it here. If you do not have one yet, skip this section and come back to it later.

These questions use the skills from Weeks 1 to 5: loading a file, checking what is missing,
making a category, and summarising with a table or a group. There is no single right answer,
because the dataset is yours.

The code cells contain templates. Replace the file name and the column names with your own,
and delete the lines you do not need.

**Task 5.1.** Load your dataset. Report how many rows and columns it has, then say in one sentence what one row represents.

Use pd.read_csv() and .shape, as in Week 3.

In [ ]:
# my_data = pd.read_csv("my_file.csv")
# print(my_data.shape)

**Task 5.2.** Choose one categorical column. Report how many of its values are missing, and what percentage of the rows that is.

Use .isna().sum() for the count and len() for the denominator, as in Week 4.

In [ ]:
# missing = my_data["my_column"].isna().sum()
# print(missing)

**Task 5.3.** Convert that column to a category and make a frequency table. Name its most common label, and say what the denominator would be if you reported percentages.

Use .astype("category") and .value_counts(), then .idxmax() for the mode.

In [ ]:
# my_data["my_column"] = my_data["my_column"].astype("category")
# print(my_data["my_column"].value_counts())

**Task 5.4.** Choose one of these two, whichever suits your question. Either cross-tabulate your column against a second categorical column with totals, or group by your column and summarise a numerical column.

Both patterns are in Section 2 and Section 3 of this worksheet.

In [ ]:
# Option A: pd.crosstab(...)
# Option B: my_data.groupby("my_column")["my_number"].agg([...])

**Task 5.5.** Write two sentences about your result. One should state a finding, naming the counting unit. The other should state something the result cannot establish.

*Write two sentences here.*

**Task 5.6.** Who created the categories in your column, and how would a different classification change your answer?

*Write one or two sentences here.*